In [3]:
import pandas as pd 

In [ ]:
def load_nasa_power(file_path="nasa_power_dakahlia.csv"):
    # Read the file, skipping the header section
    df = pd.read_csv(file_path, skiprows=12)
    
    # Clean column names
    df.columns = [col.strip() for col in df.columns]
    
    # Keep only the useful columns
    months = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
    
    # Melt the data to long format
    df_long = df.melt(
        id_vars=['PARAMETER', 'YEAR'],
        value_vars=months,
        var_name='MONTH',
        value_name='VALUE'
    )
    
    # Pivot so each parameter becomes a column
    df_wide = df_long.pivot_table(
        index=['YEAR', 'MONTH'],
        columns='PARAMETER',
        values='VALUE'
    ).reset_index()
    
    # Optional: create a proper date column
    month_map = {
        'JAN':1, 'FEB':2, 'MAR':3, 'APR':4, 'MAY':5, 'JUN':6,
        'JUL':7, 'AUG':8, 'SEP':9, 'OCT':10, 'NOV':11, 'DEC':12
    }
    df_wide['MONTH_NUM'] = df_wide['MONTH'].map(month_map)
    df_wide = df_wide.sort_values(['YEAR', 'MONTH_NUM']).reset_index(drop=True)
    
    return df_wide



df = load_nasa_power()
print(df.head(15))
print("\nColumns:", df.columns.tolist())
print("\nShape:", df.shape)

PARAMETER  YEAR MONTH  ALLSKY_SFC_SW_DWN  PRECTOTCORR   RH2M    T2M  MONTH_NUM
0          2020   JAN              10.25         1.26  71.83  12.88          1
1          2020   FEB              13.75         0.60  70.17  13.86          2
2          2020   MAR              17.97         2.74  65.75  16.30          3
3          2020   APR              23.06         3.84  62.93  19.02          4
4          2020   MAY              26.58         0.00  58.87  23.34          5
5          2020   JUN              28.48         0.00  49.51  27.40          6
6          2020   JUL              27.53         0.00  52.91  29.58          7
7          2020   AUG              25.44         0.00  54.99  29.82          8
8          2020   SEP              21.05         0.00  57.86  29.27          9
9          2020   OCT              16.55         0.03  59.86  25.97         10
10         2020   NOV              11.61         0.92  64.94  19.75         11
11         2020   DEC              10.71         0.0

In [10]:
df.to_csv('nasa_power_dakahlia_data.csv', index=True)

In [19]:
df = pd.read_csv('nasa_power_dakahlia_data.csv', index_col=0)
df.count()

YEAR                 60
MONTH                60
ALLSKY_SFC_SW_DWN    60
PRECTOTCORR          60
RH2M                 60
T2M                  60
MONTH_NUM            60
dtype: int64

In [20]:
# ============================================
# 1. Crop Information
# ============================================
CROPS = {
    "Wheat": {
        "type": "cereal",
        "season": "winter",
        "ideal_temp": (12, 20),
        "water_need_mm": 450,          # approximate seasonal need
        "n_fixation": False,
        "notes": "Common winter cereal in the Delta"
    },
    "Maize": {
        "type": "cereal",
        "season": "summer",
        "ideal_temp": (22, 32),
        "water_need_mm": 550,
        "n_fixation": False,
        "notes": "Heavy feeder, high water demand"
    },
    "Rice": {
        "type": "cereal",
        "season": "summer",
        "ideal_temp": (25, 35),
        "water_need_mm": 900,
        "n_fixation": False,
        "notes": "Very high water requirement"
    },
    "Berseem": {
        "type": "legume",
        "season": "winter",
        "ideal_temp": (12, 22),
        "water_need_mm": 350,
        "n_fixation": True,
        "notes": "Excellent nitrogen fixer and soil improver"
    },
    "Faba Bean": {
        "type": "legume",
        "season": "winter",
        "ideal_temp": (10, 20),
        "water_need_mm": 300,
        "n_fixation": True,
        "notes": "Good nitrogen fixer, common in rotations"
    }
}

In [21]:
# ============================================
# 2. Rotation Rules (Previous crop → Bonus)
# ============================================
def get_rotation_score(previous_crop, candidate):
    prev_type = CROPS[previous_crop]["type"]
    cand_type = CROPS[candidate]["type"]

    # Best practice: Cereal → Legume
    if prev_type == "cereal" and cand_type == "legume":
        return 90
    # Legume → Cereal is also good
    if prev_type == "legume" and cand_type == "cereal":
        return 85
    # Same type is less ideal
    if prev_type == cand_type:
        return 40
    # Default
    return 60

In [22]:
# ============================================
# 3. Climate Suitability Score
# ============================================
def climate_score(crop_name, avg_temp):
    ideal_min, ideal_max = CROPS[crop_name]["ideal_temp"]
    
    if ideal_min <= avg_temp <= ideal_max:
        return 95
    elif abs(avg_temp - (ideal_min + ideal_max)/2) < 5:
        return 75
    else:
        return 45

In [23]:
# ============================================
# 4. Water Balance Score
# ============================================
def water_score(crop_name, recent_rain_mm):
    need = CROPS[crop_name]["water_need_mm"]
    
    # Very simple logic for Delta (irrigation is common)
    if need < 400:
        return 90          # low water crops are better
    elif need < 600:
        return 70
    else:
        return 40          # rice is water hungry

In [24]:
# ============================================
# 5. Main Scoring Function
# ============================================
def recommend_rotations(previous_crop, climate_df, top_n=4):
    # Calculate average temperature and rainfall from recent years
    avg_temp = climate_df["T2M"].mean()
    total_rain = climate_df["PRECTOTCORR"].mean() * 30 * 6   # rough 6-month estimate
    
    results = []

    for crop in CROPS:
        if crop == previous_crop:
            continue  # skip recommending the same crop

        # Individual scores
        clim = climate_score(crop, avg_temp)
        water = water_score(crop, total_rain)
        rotation = get_rotation_score(previous_crop, crop)
        soil = 90 if CROPS[crop]["n_fixation"] else 50

        # Weighted total score
        total = (clim * 0.35) + (water * 0.25) + (rotation * 0.25) + (soil * 0.15)

        # Generate simple explanation
        reasons = []
        if CROPS[crop]["n_fixation"]:
            reasons.append("Fixes nitrogen and improves soil")
        if clim >= 80:
            reasons.append("Good temperature match")
        if water >= 80:
            reasons.append("Lower water requirement")
        if rotation >= 80:
            reasons.append("Excellent rotation after " + previous_crop)

        results.append({
            "crop": crop,
            "score": round(total, 1),
            "reasons": " | ".join(reasons) if reasons else "Moderate fit",
            "type": CROPS[crop]["type"],
            "season": CROPS[crop]["season"]
        })

    # Sort by score
    results = sorted(results, key=lambda x: x["score"], reverse=True)
    return results[:top_n]

In [25]:
print("=== Testing recommendations after Maize ===")
recs = recommend_rotations("Maize", df)

for r in recs:
    print(f"{r['crop']:12} | Score: {r['score']} | {r['reasons']}")

=== Testing recommendations after Maize ===
Berseem      | Score: 74.2 | Fixes nitrogen and improves soil | Lower water requirement | Excellent rotation after Maize
Faba Bean    | Score: 74.2 | Fixes nitrogen and improves soil | Lower water requirement | Excellent rotation after Maize
Wheat        | Score: 50.8 | Moderate fit
Rice         | Score: 43.2 | Moderate fit
